# Lab 1 - Computational Foundations for Biomedical Engineering

**Topic:** Dose map and gantry-angle analysis for helical TomoTherapy QA (ArcCHECK-inspired)

**Program:** Master's in Biomedical Engineering

**Author:** Dary Luz Durango Caro

This lab is related to my master's degree work proposal, which deals with the implementation of ArcCHECK as a dosimetric quality control method for helical tomotherapy (instead of using radiochromic film). My work still has real measurements. Following the instructions in this lab, I used synthetic data that has the same shape and idea as the real data I will use later: dose maps (planned vs. measured) and a gantry angle signal recorded over time, similar to what is reported by the ArcCHECK (SNC Patient) software.

## Context

In Helical Tomotherapy, the gantry rotates continuously as the table moves, so the dose is delivered from many angles at the same time. Before treating a patient, it is necessary to check that the dose that will actually be administered matches the dose calculated by the treatment planning system. This is usually done with radiochromic film, but film has some disadvantages: it can only be used once, it needs time to prepare, scan and analyze, and errors can be introduced during that process.

ArcCHECK is a cylindrical detector with 1386 diodes arranged in a helical pattern. Because of its shape, it can measure dose from many angles of the gantry in a single acquisition and can be reused, so it could replace film for this type of quality control (QA).

Two things are usually checked:
1. The **dose distribution** measured by ArcCHECK relative to that calculated by the planning system (TPS), typically using the gamma index (a combination of dose difference and agreement distance).
2. Some **machine parameters**, such as gantry rotation speed, can be obtained from the angle log exported by ArcCHECK software.

For this lab, I built a small program that reproduces these two ideas using synthetic data: a 2D dose map (planned vs. measured) and a gantry angle signal, and use basic Python tools (functions, loops, classes, derivatives/integrals, and matrices) to analyze them.

**References**

- Langen, K. M., Papanikolaou, N., Balog, J., Crilly, R., Followill, D., Goddu, S. M., Grant, W., Olivera, G., Ramsey, C. R., & Shi, C. (2010). QA for helical tomotherapy: Report of the AAPM Task Group 148. *Medical Physics, 37*(9), 4817-4853. https://doi.org/10.1118/1.3462971
- Miften, M., Olch, A., Mihailidis, D., Moran, J. M., Pawlicki, T., Molineu, A., Li, H., Wijesooriya, K., Meeks, S., Glass, E., & Hagan, M. (2018). Tolerance limits and methodologies for IMRT measurement-based verification QA: Recommendations of AAPM Task Group No. 218. *Medical Physics, 45*(4), e53-e83. https://doi.org/10.1002/mp.12810
- Yang, B., Wong, W. K. R., Geng, H., Lam, W. W., Ho, Y. W., Kwok, W. M., Cheung, K. Y., & Yu, S. K. (2017). Filmless methods for quality assurance of Tomotherapy using ArcCHECK. *Medical Physics, 44*(1), 7-16. https://doi.org/10.1002/mp.12009

In [ ]:
# I only used numpy and matplotlib, so this notebook does not need any external file or install.
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)  # so the "random" numbers are always the same when re-running

## Data loading and initial visualization

Since I don't have real data from ArcCHECK/TomoTherapy yet, I generated two synthetic data sets that have the same format that I will use in my degree project:

- A **planned dose map**: a 64x64 matrix representing a field of view of about 21 cm (same diameter reported for the real ArcCHECK), built with a Gaussian-shaped "hot region" to look like a simple treatment target.
- A **measured dose map**: obtained from the planned one by applying a small smoothing (to simulate that a real detector does not measure a perfectly sharp dose) plus random noise, so it looks similar to what a real acquisition would show.
- A **gantry angle signal**: angle (degrees) vs time (seconds), simulating what the ArcCHECK software exports during a helical delivery, with a constant nominal speed plus some random jitter, similar to a real rotation that is not perfectly constant.



In [ ]:
def make_planned_dose(size=64, fov_cm=10.5):
    """Build a simple synthetic 'planned dose' map: a Gaussian hot region
    over a size x size grid representing a fov_cm x fov_cm field of view."""
    x = np.linspace(-fov_cm, fov_cm, size)
    y = np.linspace(-fov_cm, fov_cm, size)
    X, Y = np.meshgrid(x, y)
    dose = np.exp(-((X-1.0)**2/(2*2.0**2) + (Y+0.5)**2/(2*1.5**2)))
    dose = dose/dose.max()
    return X, Y, dose


def make_measured_dose(planned, noise_level=0.015):
    """Simulate a 'measured' version of the planned dose: a 3x3 average
    smoothing (like a detector that cannot resolve very sharp edges) plus
    random noise (like electronic/measurement noise)."""
    size = planned.shape[0]
    measured = np.zeros_like(planned)
    for i in range(size):
        for j in range(size):
            i0, i1 = max(0, i-1), min(size, i+2)
            j0, j1 = max(0, j-1), min(size, j+2)
            measured[i, j] = planned[i0:i1, j0:j1].mean()
    noise = np.random.normal(0, noise_level, planned.shape)
    measured = measured + noise
    return np.clip(measured, 0, None)


def make_gantry_signal(n_rotations=3, points_per_rotation=90, nominal_speed=12.0):
    """Simulate a gantry angle log: n_rotations full turns at nominal_speed
    (deg/s), with a bit of random jitter added at each sample."""
    n = n_rotations*points_per_rotation
    dt = (360.0/nominal_speed)/points_per_rotation
    t = np.array([i*dt for i in range(n)])
    angle = np.array([(nominal_speed*ti + np.random.normal(0, 0.3)) % 360 for ti in t])
    return t, angle


X, Y, planned_dose = make_planned_dose()
measured_dose = make_measured_dose(planned_dose)
t, gantry_angle = make_gantry_signal()

print("Planned dose map shape :", planned_dose.shape)
print("Measured dose map shape:", measured_dose.shape, "range:", measured_dose.min(), measured_dose.max())
print("Gantry signal length   :", len(t), "samples, from t=0 to t=%.1f s" % t[-1])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
extent = [-10.5, 10.5, -10.5, 10.5]

im0 = axes[0].imshow(planned_dose, cmap="jet", extent=extent, origin="lower")
axes[0].set_title("Planned dose")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(measured_dose, cmap="jet", extent=extent, origin="lower")
axes[1].set_title("Measured dose (synthetic)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

diff0 = measured_dose - planned_dose
im2 = axes[2].imshow(diff0, cmap="bwr", extent=extent, origin="lower", vmin=-0.15, vmax=0.15)
axes[2].set_title("Measured - Planned")
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 3.5))
plt.plot(t, gantry_angle, linewidth=0.8)
plt.xlabel("Time (s)")
plt.ylabel("Gantry angle (deg)")
plt.title("Simulated gantry angle log")
plt.show()

The planned map looks like a smooth "target" region, and the measured one looks similar but a bit blurry and noisy, which is what you would expect from a real detector. The angle graph goes from 0 to 360 repeatedly (because the gantry keeps rotating), which is the typical shape for this type of sign.

## OOP for data processing

I defined two classes:

- `DoseMapAnalyzer`: keeps a planned/measured dose map pair and has methods to compute the difference, the gradient (using a basic finite-difference formula I implemented with `for` loops), a simplified gamma index, and a profile line.
- `GantrySignalAnalyzer`: keeps the angle-vs-time signal and has methods to "unwrap" the angle (so it doesn't jump from 359° to 0°), compute the angular velocity and the total rotation, and check which samples are outside a speed tolerance (using a list comprehension).

In [ ]:
class DoseMapAnalyzer:
    """Stores a planned/measured dose map pair and computes some simple QA metrics."""

    def __init__(self, planned, measured, label="map1"):
        self.planned = planned
        self.measured = measured
        self.label = label

    def difference(self):
        return self.measured - self.planned

    def gradient(self):
        # basic central-difference formula: df/dx ~ (f[x+1]-f[x-1]) / 2
        rows, cols = self.planned.shape
        gx = np.zeros_like(self.planned)
        gy = np.zeros_like(self.planned)
        for i in range(1, rows-1):
            for j in range(1, cols-1):
                gx[i, j] = (self.planned[i, j+1] - self.planned[i, j-1]) / 2
                gy[i, j] = (self.planned[i+1, j] - self.planned[i-1, j]) / 2
        grad_mag = np.sqrt(gx**2 + gy**2)
        return gx, gy, grad_mag

    def gamma_index(self, dose_crit_pct=3.0, dist_crit_mm=3.0, pixel_mm=3.28, dose_thresh_pct=10.0):
        # simplified gamma index: for each point above the dose threshold, look
        # in a small neighborhood for the closest match in dose+distance
        rows, cols = self.planned.shape
        dose_norm = self.planned.max()
        search = 3  # pixels
        gamma = np.full(self.planned.shape, np.nan)
        for i in range(rows):
            for j in range(cols):
                if self.planned[i, j] < (dose_thresh_pct/100.0)*dose_norm:
                    continue
                best = None
                for di in range(-search, search+1):
                    for dj in range(-search, search+1):
                        ii, jj = i+di, j+dj
                        if 0 <= ii < rows and 0 <= jj < cols:
                            dist_mm = ((di**2 + dj**2)**0.5)*pixel_mm
                            dose_diff_pct = 100.0*abs(self.planned[ii, jj]-self.measured[i, j])/dose_norm
                            g = ((dose_diff_pct/dose_crit_pct)**2 + (dist_mm/dist_crit_mm)**2)**0.5
                            if best is None or g < best:
                                best = g
                gamma[i, j] = best
        valid = gamma[~np.isnan(gamma)]
        pass_rate = 100.0*np.mean(valid <= 1.0) if len(valid) else float("nan")
        return gamma, pass_rate

    def profile(self, row=None):
        row = self.planned.shape[0]//2 if row is None else row
        return self.planned[row, :], self.measured[row, :]

    def summary(self):
        diff = self.difference()
        return {"label": self.label,
                "mean_abs_diff": float(np.mean(np.abs(diff))),
                "max_planned": float(self.planned.max()),
                "max_measured": float(self.measured.max())}


class GantrySignalAnalyzer:
    """Stores a gantry angle vs time signal and computes speed / rotation."""

    def __init__(self, t, angle_deg, nominal_speed):
        self.t = t
        self.angle_deg = angle_deg
        self.nominal_speed = nominal_speed

    def unwrap_angle(self):
        # the raw angle jumps from ~360 back to 0 every rotation, so I "unwrap"
        # it by accumulating the small steps instead of the raw value
        unwrapped = [self.angle_deg[0]]
        for i in range(1, len(self.angle_deg)):
            step = self.angle_deg[i] - self.angle_deg[i-1]
            if step < -180:
                step += 360
            elif step > 180:
                step -= 360
            unwrapped.append(unwrapped[-1] + step)
        return np.array(unwrapped)

    def angular_velocity(self):
        # derivative of the unwrapped angle, using a central difference formula
        theta = self.unwrap_angle()
        n = len(theta)
        v = np.zeros(n)
        for i in range(1, n-1):
            v[i] = (theta[i+1]-theta[i-1]) / (self.t[i+1]-self.t[i-1])
        v[0] = (theta[1]-theta[0]) / (self.t[1]-self.t[0])
        v[-1] = (theta[-1]-theta[-2]) / (self.t[-1]-self.t[-2])
        return v

    def total_rotation(self):
        # integral of the angular velocity using the trapezoidal rule
        v = self.angular_velocity()
        total = 0.0
        for i in range(len(v)-1):
            dt = self.t[i+1]-self.t[i]
            total += 0.5*(v[i]+v[i+1])*dt
        return total

    def out_of_tolerance(self, tol_deg_s=0.6):
        v = self.angular_velocity()
        return [i for i, val in enumerate(v) if abs(val - self.nominal_speed) > tol_deg_s]


dose_analyzer = DoseMapAnalyzer(planned_dose, measured_dose, label="slice_1")
gantry_analyzer = GantrySignalAnalyzer(t, gantry_angle, nominal_speed=12.0)

print("Dose map summary:", dose_analyzer.summary())
print("Number of samples out of the speed tolerance:", len(gantry_analyzer.out_of_tolerance()))

In [ ]:
gamma_map, gamma_pass_rate = dose_analyzer.gamma_index(dose_crit_pct=3.0, dist_crit_mm=3.0)
print(f"Gamma index (3%/3mm) pass rate: {gamma_pass_rate:.2f} %")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(dose_analyzer.difference(), cmap="bwr", extent=extent, origin="lower", vmin=-0.15, vmax=0.15)
axes[0].set_title("Dose difference")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(gamma_map, cmap="viridis", extent=extent, origin="lower", vmin=0, vmax=2)
axes[1].set_title(f"Gamma map (pass rate = {gamma_pass_rate:.1f}%)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

The gamma pass rate is high but not 100%, which makes sense because I added smoothing and noise on purpose. The points that fail are mostly located near the edge of the "target" region, where the dose changes quickly from high to low (I check this in the next section with the gradient).

## Numerical derivation or integration of data

I applied a derivative and an integral to my two datasets:

- **Derivative on the dose map:** I used the `gradient()` method of `DoseMapAnalyzer` (central-difference formula, done with `for` loops) to compute how fast the dose changes spatially. This highlights the edge of the target region.
- **Derivative and integral on the gantry signal:** I used `angular_velocity()` (derivative of the angle) and `total_rotation()` (integral of the velocity, trapezoidal rule) from `GantrySignalAnalyzer`. If everything is consistent, integrating the velocity back should give approximately the total angle rotated (3 full turns = 1080°).

In [ ]:
gx, gy, grad_mag = dose_analyzer.gradient()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(planned_dose, cmap="jet", extent=extent, origin="lower")
axes[0].set_title("Planned dose")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(grad_mag, cmap="inferno", extent=extent, origin="lower")
axes[1].set_title("Gradient magnitude (edge of the target)")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
v = gantry_analyzer.angular_velocity()
total_rot = gantry_analyzer.total_rotation()
expected_rot = 3*360

print(f"Mean angular velocity: {v.mean():.2f} deg/s (nominal = 12.0 deg/s)")
print(f"Total rotation (integrated back from the velocity): {total_rot:.1f} deg")
print(f"Expected total rotation: {expected_rot} deg  ->  error = {100*(total_rot-expected_rot)/expected_rot:.2f} %")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(t, v, linewidth=0.8)
axes[0].axhline(12.0, color="k", linestyle="--", linewidth=1, label="Nominal speed")
axes[0].set_xlabel("Time (s)"); axes[0].set_ylabel("Angular velocity (deg/s)")
axes[0].set_title("Angular velocity (derivative)")
axes[0].legend()

theta_unwrapped = gantry_analyzer.unwrap_angle()
axes[1].plot(t, theta_unwrapped, linewidth=0.8)
axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Unwrapped angle (deg)")
axes[1].set_title("Unwrapped angle (reference, not derived/integrated)")
plt.tight_layout()
plt.show()

The angular velocity oscillates around the nominal speed of 12 deg/s, which is expected because I added random jitter to the signal. When I integrate the velocity back with the trapezoidal rule, I get a total rotation very close to the expected 1080° (3 full turns), with a small error caused by the discrete derivative/integral and the noise - this is basically the same consistency check that the ArcCHECK software would do to detect if the gantry speed was not stable.

## Matrix operations

I performed three matrix/tensor operations on the dose map data:

1. **Tensor slicing:** I repeated the measurement 5 times (with different random noise each time) and stacked the results into a 3D array (a tensor), then sliced it to get one full repeat and one column across all repeats. This is related to the "repeatability" check mentioned in my thesis protocol.
2. **Matrix multiplication:** I built a diagonal matrix representing a row-by-row sensitivity correction (similar to what a calibration correction would do to each row of diodes) and multiplied it by the measured dose matrix.
3. **Eigenvalues:** I built the covariance matrix between the 5 repeated measurements and computed its eigenvalues, which give an idea of how much the repeated measurements vary from each other.

In [ ]:
# 1) Tensor of repeated measurements + slicing
n_repeats = 5
repeats = np.array([make_measured_dose(planned_dose) for _ in range(n_repeats)])
print("Tensor shape (repeats, rows, cols):", repeats.shape)

mean_map = repeats.mean(axis=0)
std_map = repeats.std(axis=0)
repeat_2 = repeats[1, :, :]           # slicing: one full repeat
central_column = repeats[:, :, 32]    # slicing: one column across all repeats
print("Central column across repeats, shape:", central_column.shape)
print("Max standard deviation across repeats:", std_map.max())

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(mean_map, cmap="jet", extent=extent, origin="lower")
axes[0].set_title(f"Mean of {n_repeats} repeats")
plt.colorbar(im0, ax=axes[0], fraction=0.046)
im1 = axes[1].imshow(std_map, cmap="inferno", extent=extent, origin="lower")
axes[1].set_title("Std. dev. across repeats")
plt.colorbar(im1, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
# 2) Matrix multiplication: diagonal "sensitivity correction" matrix
rows, cols = measured_dose.shape
sensitivity_values = 1 + 0.02*np.sin(np.linspace(0, 2*np.pi, rows))
D = np.diag(sensitivity_values)          # diagonal matrix, one value per row
corrected_dose = D @ measured_dose       # matrix multiplication
print("Corrected dose range:", corrected_dose.min(), corrected_dose.max())

# 3) Eigenvalues of the covariance matrix between repeats
flat = repeats.reshape(n_repeats, -1)
flat_centered = flat - flat.mean(axis=0, keepdims=True)
cov_matrix = np.cov(flat_centered)
eigvals, eigvecs = np.linalg.eigh(cov_matrix)
print("Eigenvalues of the covariance matrix:", np.round(eigvals, 6))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(corrected_dose, cmap="jet", extent=extent, origin="lower")
axes[0].set_title("Dose after row-wise\nsensitivity correction (D @ measured)")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

axes[1].bar(range(1, len(eigvals)+1), sorted(eigvals, reverse=True))
axes[1].set_xlabel("Component")
axes[1].set_ylabel("Eigenvalue")
axes[1].set_title("Eigenvalues of the covariance\nbetween repeated measurements")
plt.tight_layout()
plt.show()

## Discussion

The results make sense because of the way I constructed the synthetic data. The gamma index pass rate (around 90%) is high because the "measured" map is basically the planned map plus a little blur and noise, but it's not 100% because the failing points are mostly around the edge of the target, which I confirmed by looking at the gradient magnitude map: the gradient is highest exactly where the gamma map shows the most failed points, which is expected because the dose difference criteria are harder to pass where the dose changes quickly.

For the gantry signal, the angular velocity hovers around the nominal 12 deg/s because I added random noise to the angle. When I integrate this speed again with the trapezoidal rule, I recover a total rotation very close to the expected value (3 turns = 1080°), with only a small error. This type of derivative/integral consistency check is basically what the actual ArcCHECK Gantry QA module does to verify that the gantry rotates at a stable speed.

The matrix/tensor part shows three different but related ideas: Stacking repeated measurements into a 3D matrix and dividing them allows me to calculate a mean and standard deviation per pixel, which is a simple way to check repeatability. Matrix multiplication by a diagonal matrix is ​​a simple example of how a row correction (such as a calibration factor) could be applied to a real dose map. And the eigenvalues ​​of the covariance matrix between repetitions give an idea of ​​how much variability there is between measurements, without having to look at each pixel manually.

Since this is synthetic data, I cannot draw real dosimetric conclusions from it: the goal of this lab was to build and test a small computational process (classes, derivative/integral, matrix operations) that I can later reuse with real data from ArcCHECK/TomoTherapy.

## Conclusions

I developed a small Python pipeline (functions and two classes) to analyze dose maps and a gantry angle signal, using structured synthetic data. The gamma index calculation yielded a high overall pass rate, with the few failed points concentrated near the edges of the dose region; this spatial pattern was consistent with the gradient (derivative) analysis, suggesting that the discrepancies are related to regions of strong dose variation rather than random error. Similarly, the derivative and integral of the gantry angle signal were mutually consistent, and the total rotation recovered was close to the expected 1080°, indicating that this type of derivative/integral cross-checking could be a useful tool for detecting gantry velocity anomalies in real acquisitions. The matrix and tensor component of the pipeline, which involves stacking repeated measurements, matrix multiplication, and calculating eigenvalues ​​from the covariance matrix, provided simple but effective ways to evaluate reproducibility and apply a basic correction to a dose map. Finally, since the entire process is based solely on numpy and matplotlib and was tested using synthetically generated data, the notebook is completely self-contained and the same classes are expected to work directly with real data once it is obtained.

## AI Use Declaration

I used Claude (Anthropic) as a support tool while working on this notebook. He helped me organize the notebook into the sections requested in the guide, explained several Python concepts I wasn't previously familiar with (such as unwrapping an angle signal and writing a simplified gamma index), and helped me debug parts of the code that weren't working correctly. I reviewed, tested and modified all the code myself.